In [2]:
pip install laya datasets numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 977.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.1/286.1 kB 3.0 MB/s eta 0:00:00


In [4]:
"""
Laya decision model on an existing dataset (DAIR Emotion, 6 labels).

Setup:
    pip install laya datasets numpy

What it does:
  1. Loads the test split of dair-ai/emotion.
  2. Asks Laya one typed `choice` question per text (batched).
  3. Reports accuracy and ECE (calibration).
  4. Applies confidence gating: auto-decide if confidence >= threshold,
     otherwise escalate. Prints accuracy vs coverage per threshold.

Note: base Laya is a zero-shot-weak model; expect modest accuracy
(~0.57 reported on this dataset). Fine-tuning is where the gains are.
"""
import numpy as np
import laya
from datasets import load_dataset

N_SAMPLES = 500          # subsample for a quick run; set None for full test set
BATCH_SIZE = 64
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]

LABELS = ["sadness", "joy", "love", "anger", "fear", "surprise"]  # dataset order

QUESTIONS = {
    "emotion": {
        "type": "choice",
        "instructions": "Which emotion does the author express?",
        "criteria": {
            "sadness": "sad, grief, loss, feeling down",
            "joy": "happy, delighted, excited, content",
            "love": "affection, caring, fondness, romantic feelings",
            "anger": "angry, irritated, resentful, furious",
            "fear": "afraid, anxious, worried, scared",
            "surprise": "surprised, shocked, astonished",
        },
    }
}


def expected_calibration_error(conf, correct, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece


def main():
    ds = load_dataset("dair-ai/emotion", split="test")
    if N_SAMPLES:
        ds = ds.shuffle(seed=0).select(range(N_SAMPLES))

    texts = list(ds["text"])
    gold = np.array([LABELS[i] for i in ds["label"]])

    # English checkpoint (ModernBERT-large, 512 ctx). Add device="cuda" if available.
    agent = laya.load("convaiinnovations/laya")

    states = [{"body": t} for t in texts]
    results = agent.predict_batch(states, QUESTIONS, batch_size=BATCH_SIZE)

    pred = np.array([r["answers"]["emotion"]["choice"] for r in results])
    conf = np.array([r["answers"]["emotion"]["confidence"] for r in results])
    correct = (pred == gold).astype(float)

    print(f"Samples        : {len(texts)}")
    print(f"Accuracy       : {correct.mean():.3f}   (random = {1/len(LABELS):.3f})")
    print(f"Mean confidence: {conf.mean():.3f}")
    print(f"ECE            : {expected_calibration_error(conf, correct):.3f}")

    print("\nConfidence gating (auto-decide vs escalate):")
    print(f"{'thr':>5} {'coverage':>9} {'acc@auto':>9} {'escalated':>10}")
    for t in THRESHOLDS:
        auto = conf >= t
        cov = auto.mean()
        acc = correct[auto].mean() if auto.any() else float("nan")
        print(f"{t:>5.2f} {cov:>9.3f} {acc:>9.3f} {(~auto).sum():>10d}")


if __name__ == "__main__":
    main()

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

...

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:1869: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Samples        : 500
Accuracy       : 0.590   (random = 0.167)
Mean confidence: 0.768
ECE            : 0.210

Confidence gating (auto-decide vs escalate):
  thr  coverage  acc@auto  escalated
 0.50     0.828     0.633         86
 0.60     0.742     0.658        129
 0.70     0.664     0.681        168
 0.80     0.594     0.690        203
 0.90     0.488     0.709        256
